In [ ]:
import pandas as pd
import re
import sys
from pathlib import Path

sys.path.append(str(Path().resolve().parent))


import pandas as pd
import os
from config import SILVER_DATA_DIR,BRONZE_DATASET
OUT_FILE = SILVER_DATA_DIR / "silver_databridge.csv"

df = pd.read_csv(BRONZE_DATASET)

print("Paths configurados ✅")

Paths configurados ✅


In [ ]:
df = pd.read_csv(BRONZE_DATASET, dtype=str)  # mantém tudo string até tiparmos explicitamente

print(f"Linhas  : {len(df):,}")
print(f"Colunas : {df.shape[1]}")
df.head(2)

Linhas  : 30,000
Colunas : 62


,application_id,source_system,ocr_engine,doc_type,page_count,file_size_kb,image_blur,skew_angle_deg,document_image_quality,ocr_confidence,...,credit_requested_value,income_declared,tenure_months,collateral_type,ltv,pd_model_score,final_decision,reason_code,requires_human_review,default_12m
0,APP-20260000001,OCR_PDF,azure_ocr,EXTRATO,1,1421.0,0.14,-1.35,0.821,0.562,...,10898.86,53421.8,210,IMOVEL,0.67,0.297,APPROVE,OK,1,0
1,APP-20260000002,OCR_PDF,tesseract,RG,9,1556.0,0.016,0.5,0.968,0.689,...,19605.0,22433.06,125,IMOVEL,1.63,0.346,REVIEW,HIGH_RISK,1,0


In [ ]:
# ── Inteiros ────────────────────────────────────────────────────────────────
int_cols = [
    "page_count", "ocr_error_count", "version", "rule_violations",
    "pii_detected", "fire_hotspots_30d", "days_rain_30d",
    "tenure_months", "requires_human_review", "default_12m"
]
for c in int_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce").astype("Int64")  # Int64 suporta nulos

# ── Floats ──────────────────────────────────────────────────────────────────
float_cols = [
    "image_blur", "skew_angle_deg", "document_image_quality", "ocr_confidence",
    "normalized_amount", "match_score", "data_quality_score",
    "deforestation_km2_12m", "lat", "lon",
    "precip_mm_30d", "temp_c_mean_30d", "drought_spi", "flood_risk_idx",
    "ndvi", "soil_moisture", "wind_speed_mean", "humidity_mean",
    "credit_requested_value", "income_declared", "ltv", "pd_model_score"
]
for c in float_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

print("Tipagem numérica concluída ✅")
df[int_cols + float_cols].dtypes

Tipagem numérica concluída ✅


page_count                  Int64
ocr_error_count             Int64
version                     Int64
rule_violations             Int64
pii_detected                Int64
fire_hotspots_30d           Int64
days_rain_30d               Int64
tenure_months               Int64
requires_human_review       Int64
default_12m                 Int64
image_blur                float64
skew_angle_deg            float64
document_image_quality    float64
ocr_confidence            float64
normalized_amount         float64
match_score               float64
data_quality_score        float64
deforestation_km2_12m     float64
lat                       float64
lon                       float64
precip_mm_30d             float64
temp_c_mean_30d           float64
drought_spi               float64
flood_risk_idx            float64
ndvi                      float64
soil_moisture             float64
wind_speed_mean           float64
humidity_mean             float64
credit_requested_value    float64
income_declare

In [ ]:
df["normalized_date"] = pd.to_datetime(df["normalized_date"], errors="coerce")
df["created_at"]      = pd.to_datetime(df["created_at"],      errors="coerce")
df["updated_at"]      = pd.to_datetime(df["updated_at"],      errors="coerce")

print("Tipagem de datas concluída ✅")
df[["normalized_date", "created_at", "updated_at"]].dtypes

Tipagem de datas concluída ✅


normalized_date    datetime64[us]
created_at         datetime64[us]
updated_at         datetime64[us]
dtype: object

In [ ]:
def clean_file_size_kb(series: pd.Series) -> pd.Series:
    """Remove sufixo 'KB' (drift ~3%) e converte para float."""
    return (
        series
        .str.replace(r"\s*KB\s*", "", regex=True, flags=re.IGNORECASE)
        .str.strip()
        .pipe(pd.to_numeric, errors="coerce")
    )

antes = df["file_size_kb"].str.contains("KB", na=False).sum()
df["file_size_kb"] = clean_file_size_kb(df["file_size_kb"])

print(f"Linhas corrigidas (sufixo KB): {antes:,}")
print(f"Nulos restantes em file_size_kb: {df['file_size_kb'].isna().sum()}")
df["file_size_kb"].describe()

Linhas corrigidas (sufixo KB): 887
Nulos restantes em file_size_kb: 0


count    30000.000000
mean       857.061333
std        393.109340
min         50.000000
25%        583.000000
50%        852.000000
75%       1123.000000
max       2853.000000
Name: file_size_kb, dtype: float64

In [ ]:
n_nulos_antes = df["ocr_confidence"].isna().sum()

mediana_por_engine = df.groupby("ocr_engine")["ocr_confidence"].transform("median")
df["ocr_confidence"] = df["ocr_confidence"].fillna(mediana_por_engine)

n_nulos_depois = df["ocr_confidence"].isna().sum()
print(f"Nulos antes : {n_nulos_antes:,}")
print(f"Nulos depois: {n_nulos_depois:,}")
df.groupby("ocr_engine")["ocr_confidence"].agg(["mean", "median", "count"])

Nulos antes : 1,440
Nulos depois: 0


,mean,median,count
ocr_engine,,,
azure_ocr,0.699374,0.7110,10189
google_vision,0.702059,0.7145,9946
tesseract,0.700355,0.7120,9865


In [ ]:
str_cols = [
    "source_system", "ocr_engine", "doc_type", "text_language",
    "join_status", "normalization_method", "compliance_status",
    "retention_class", "bioma", "customer_segment", "industry_sector",
    "collateral_type", "final_decision", "reason_code",
    "climate_alert_level", "regiao", "uf"
]
for c in str_cols:
    df[c] = df[c].str.strip().str.upper()

print("Strings padronizadas ✅")
print("\nExemplo — final_decision:")
print(df["final_decision"].value_counts())

Strings padronizadas ✅

Exemplo — final_decision:
final_decision
APPROVE    26156
REVIEW      3844
Name: count, dtype: int64


In [ ]:
df["is_duplicate"] = df["duplicate_group_id"].notna().astype(int)

print(f"Registros marcados como duplicatas: {df['is_duplicate'].sum():,} ({df['is_duplicate'].mean():.2%})")

Registros marcados como duplicatas: 300 (1.00%)


In [ ]:
df["env_risk_flag"] = (
    (df["drought_spi"]        < -1.0).astype(int) +
    (df["flood_risk_idx"]     >  0.6).astype(int) +
    (df["fire_hotspots_30d"]  >  5  ).astype(int)
)

df["env_risk_level"] = pd.cut(
    df["env_risk_flag"],
    bins=[-1, 0, 1, 3],
    labels=["BAIXO", "MEDIO", "ALTO"]
)

print("Distribuição do risco ambiental composto:")
print(df["env_risk_level"].value_counts())

Distribuição do risco ambiental composto:
env_risk_level
BAIXO    22510
MEDIO     7126
ALTO       364
Name: count, dtype: int64


In [ ]:
numeric_cols = int_cols + float_cols
nulls_restantes = df[numeric_cols].isnull().sum()

print("Nulos restantes nas colunas numéricas:")
display(nulls_restantes[nulls_restantes > 0].rename("null_count").to_frame())

print(f"\nLinhas totais  : {len(df):,}")
print(f"Colunas totais : {df.shape[1]}")

Nulos restantes nas colunas numéricas:


,null_count



Linhas totais  : 30,000
Colunas totais : 65


In [ ]:
# Resumo das transformações aplicadas
resumo = {
    "file_size_kb corrigidos (KB)": df["file_size_kb"].notna().sum(),
    "ocr_confidence imputados": n_nulos_antes,
    "is_duplicate flagados": df["is_duplicate"].sum(),
    "env_risk_level ALTO": (df["env_risk_level"] == "ALTO").sum(),
    "env_risk_level MEDIO": (df["env_risk_level"] == "MEDIO").sum(),
    "env_risk_level BAIXO": (df["env_risk_level"] == "BAIXO").sum(),
}

pd.Series(resumo, name="count").to_frame()

,count
file_size_kb corrigidos (KB),30000
ocr_confidence imputados,1440
is_duplicate flagados,300
env_risk_level ALTO,364
env_risk_level MEDIO,7126
env_risk_level BAIXO,22510


In [ ]:
df = df.drop(columns=[
    "application_id",
    "master_id",
    "checksum",
    "lineage_path",
    "duplicate_group_id",
    "raw_amount_text",
    "raw_date_text",
    "free_text",
    "ingest_batch_id",
    "updated_at",
    "created_at",
    "reason_code",
    "join_status",
    "version",
    "file_size_kb",
    "lat",
    "lon",
    "image_blur",
    "skew_angle_deg",
    "retention_class",
    "page_count",
    "env_risk_flag",
    "wind_speed_mean",
    "humidity_mean",
    "temp_c_mean_30d",
    "days_rain_30d",
    "soil_moisture",
    "source_system"
])

In [ ]:
df = df[df["doc_type"] != "RG"]

In [ ]:
df.head()

,ocr_engine,doc_type,document_image_quality,ocr_confidence,ocr_error_count,text_language,normalized_amount,normalized_date,normalization_method,match_score,...,income_declared,tenure_months,collateral_type,ltv,pd_model_score,final_decision,requires_human_review,default_12m,is_duplicate,env_risk_level
0,AZURE_OCR,EXTRATO,0.821,0.562,4,PT,10348.78,2024-06-19,RULES_V1,0.720,...,53421.80,210,IMOVEL,0.67,0.297,APPROVE,1,0,0,BAIXO
2,GOOGLE_VISION,EXTRATO,0.622,0.830,1,PT,21423.14,2025-09-17,RULES_V2,0.659,...,28138.01,152,VEICULO,0.67,0.223,APPROVE,0,1,0,BAIXO
3,GOOGLE_VISION,CONTRATO,0.430,0.649,3,PT,44288.16,2025-03-19,RULES_V2,0.830,...,41331.41,213,CPR,2.00,0.330,REVIEW,1,1,0,BAIXO
4,GOOGLE_VISION,EXTRATO,0.787,0.673,5,PT,6021.65,2025-11-02,ML_V1,0.749,...,25293.87,99,SEM_GARANTIA,0.50,0.338,APPROVE,0,0,0,BAIXO
7,GOOGLE_VISION,NF,0.646,0.499,5,PT,7328.61,2025-05-01,ML_V1,0.585,...,19692.55,148,IMOVEL,0.29,0.284,APPROVE,1,0,0,BAIXO


In [ ]:
df.to_csv(OUT_FILE, index=False)
size_mb = os.path.getsize(OUT_FILE) / 1024 / 1024
print(f"✅ Silver salvo em: {OUT_FILE}")
print(f"   Tamanho        : {size_mb:.2f} MB")
print(f"   Linhas         : {len(df):,}")
print(f"   Colunas        : {df.shape[1]}")

✅ Silver salvo em: C:\Users\Marce\OneDrive\Área de Trabalho\bb_2\data\silver/silver_databridge.csv
   Tamanho        : 4.97 MB
   Linhas         : 24,974
   Colunas        : 37
